# AE_reproduce_figures_tables: every figure and table of the submission

**What it does.** Rebuilds, from the analysis outputs, every figure and table of the manuscript and its additional files, and puts
them in `output_reproduction/` under the names used in the submission: Figures 1-7, Figures S1-S6 (Additional file 3), Tables
S1-S33 (Additional file 2), the figure legends, every PIC number of the text and the visual abstract. A step that has not been run in
this folder is run first, in order. If the submitted package is in the folder, each result is compared with it.

**Folder layout.** One folder holds everything (a short path with ASCII characters only, for example `D:\Sub_ANE`):

```
Sub_ANE\
    AE_reproduce_figures_tables.ipynb        this notebook (Run All)
    AE_subphenotypes_analysis.ipynb          derivation analysis
    AE_subphenotypes_redraw_figures.ipynb    redraws its figures and supplementary files from the saved run
    AE_export_frozen_model.ipynb             optional: rebuilds the frozen model from the saved run (not needed here)
    requirements.txt
    AE_freeze_v1.4\                          frozen dataset (child-level; stays on this computer)
    chart_review_completed.xlsx              completed chart-review sheet (stays on this computer)
    pic_external_validation\
        PIC_SUB_01_feasibility.ipynb ... PIC_SUB_04_manuscript_outputs.ipynb
        PIC_SUB_02_frozen_wuhan_model.json
        PIC_SUB_02_config.txt                from PIC_SUB_02_config_template.txt: OSF link, and pic_dir if the PIC files are elsewhere
        requirements_pic.txt
        PIC version 1.1.0 table files (.csv), or a folder named by pic_dir (stay on this computer)
    Critical_Care_submission_20261020.zip    optional: the submitted package, for the comparison
```

The runs then add `cache_frozen_tables\`, `cache_figure_state\` and `output_subphenotypes\` (derivation), the `output_pic_sub*`
folders in `pic_external_validation\` (PIC) and `output_reproduction\` (this notebook). If the analyses have already been run
elsewhere, copying their output folders into this layout (`cache_figure_state\`, `cache_frozen_tables\`, `output_subphenotypes\`,
and `output_pic_sub02\`, `output_pic_sub02_LOCAL_ONLY\`, `output_pic_sub03_posthoc\`, `output_pic_sub03_posthoc_LOCAL_ONLY\`) is
enough: they are used as they are and not run again.

**Steps**, each only when needed:

1. Derivation analysis, `AE_subphenotypes_analysis.ipynb`, when no saved run is in the folder (hours).
2. PIC runs, `PIC_SUB_02_check`, `PIC_SUB_02_replication` and `PIC_SUB_03_posthoc`, when their outputs are missing (minutes; they
   read the PIC files). `PIC_SUB_01_feasibility` documents the counts made before the addendum and is not needed for any result.
3. Figures 1-6 and S1-S6, Tables S1-S26 and the legends, redrawn from the saved run by `AE_subphenotypes_redraw_figures.ipynb`
   (about a minute; skipped when step 1 has just drawn them).
4. Figure 7 and its legend, every PIC number of the text and Tables S27-S33, by `PIC_SUB_04_manuscript_outputs.ipynb` from the exact
   PIC tables, appended to Tables S1-S26 of step 3 (under a minute). It runs in a temporary folder, so the PIC folder is unchanged.
5. Visual abstract: an editable slide (`Visual_Abstract_920x300.pptx`, python-pptx) and its 920 x 300 pixel PNG (LibreOffice or,
   on Windows, PowerPoint).
6. Collection in `output_reproduction/` and comparison with the submitted package.

**How to run.** In the environment of the derivation analysis (`requirements.txt`; optional: `python-pptx` for the visual abstract
and `pymupdf` for side-by-side images of the figures):
*Restart Kernel and Run All Cells*, or, for a run that includes step 1, from the command line:

```
jupyter nbconvert --to notebook --execute AE_reproduce_figures_tables.ipynb --output-dir output_reproduction --ExecutePreprocessor.timeout=-1
```

The PIC analyses of record ran in their own environment (`requirements_pic.txt`, Python 3.12). To run steps 2 and 4 there, set
`PIC_PYTHON` in the next cell to the Python of that environment. Close any output file (PDF, Excel) that is open before running.

**Results that can differ on another computer.** A few results depend on whether model refits converge, which can vary with the
computer and the library versions: for H4 and E1 (Tables S19 and S18, and Table S10), the convergence flag, the number of bootstrap
replicates used, the third decimal of the bootstrap CI of the change in C statistic, the number of parametric-bootstrap simulations
and whether the parametric bootstrap is reported; and the same items in the legends of Figures 4 and 5. The comparison marks these as
resampling-dependent and lists every other difference separately.

**Sharing.** The figures, tables, legends and numbers in `output_reproduction/` are aggregate results that the manuscript reports.
The executed copies of the notebooks (`output_reproduction/executed_notebooks/`) are records of the runs, like their logs, and are not
uploaded. The dataset, the PIC files, `cache_*` folders and `*_LOCAL_ONLY` folders stay on this computer.

In [ ]:
# Settings (change only if needed)
PIC_PYTHON = None          # Python of the PIC environment, e.g. r'C:\Users\me\miniconda3\envs\ae_pic\python.exe'; None = this one
RERUN_DERIVATION = False   # True: run the derivation analysis again even if a saved run is in the folder (hours)
RERUN_PIC = False          # True: run PIC_SUB_02_check, PIC_SUB_02 and PIC_SUB_03 again even if their outputs are in the folder
VISUAL_ABSTRACT = True     # build the visual abstract (python-pptx; LibreOffice, or PowerPoint on Windows, for the PNG)

import difflib, hashlib, io, json, math, os, platform, re, shutil, subprocess, sys, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import openpyxl

T_START = time.time()
HERE = Path.cwd()
PIC = HERE / 'pic_external_validation'
OUT = HERE / 'output_reproduction'
EXEC = OUT / 'executed_notebooks'
WORK = HERE / '_reproduction_work_LOCAL_ONLY'            # temporary copy of the PIC tables for step 4; removed at the end
ANALYSIS_NB = HERE / 'AE_subphenotypes_analysis.ipynb'
REDRAW_NB = HERE / 'AE_subphenotypes_redraw_figures.ipynb'
PIC_NBS = {k: PIC / f'{k}.ipynb' for k in ('PIC_SUB_02_check', 'PIC_SUB_02_replication', 'PIC_SUB_03_posthoc',
                                           'PIC_SUB_04_manuscript_outputs')}
MODEL = PIC / 'PIC_SUB_02_frozen_wuhan_model.json'
MODEL_SHA = 'a8a51287d7dd2cb67b623328ffedc663deb9a01b3c5364db3c4e9d8f39be2179'   # the file deposited on OSF

_missing = [p.relative_to(HERE).as_posix() for p in [ANALYSIS_NB, REDRAW_NB, MODEL, *PIC_NBS.values()] if not p.is_file()]
if _missing:
    raise FileNotFoundError('Not found: ' + ', '.join(_missing) + '. Put this notebook in the repository folder (layout in the first cell).')
if hashlib.sha256(MODEL.read_bytes()).hexdigest() != MODEL_SHA:
    raise RuntimeError('pic_external_validation/PIC_SUB_02_frozen_wuhan_model.json differs from the file deposited on OSF.')

FREEZE = HERE / 'AE_freeze_v1.4'
if not (FREEZE / 'manifest.json').exists() and (FREEZE / 'AE_freeze_v1.4' / 'manifest.json').exists():
    FREEZE = FREEZE / 'AE_freeze_v1.4'
SAVED_RUN = ((HERE / 'cache_figure_state' / 'figure_state_INTERNAL.pkl').is_file()
             and (HERE / 'output_subphenotypes' / 'code' / 'ae_lpa_tools.py').is_file())
PIC_OUT = {'sub02_exact': PIC / 'output_pic_sub02_LOCAL_ONLY', 'sub02': PIC / 'output_pic_sub02',
           'sub03_exact': PIC / 'output_pic_sub03_posthoc_LOCAL_ONLY', 'sub03': PIC / 'output_pic_sub03_posthoc'}
PIC_DONE = all((PIC_OUT[k] / 'run_info.json').is_file() for k in ('sub02_exact', 'sub02', 'sub03_exact'))
PIC_CONFIG = PIC / 'PIC_SUB_02_config.txt'


def find_submitted():
    """The submitted package next to this notebook: an unzipped folder or the zip file (newest first)."""
    for p in sorted(HERE.glob('Critical_Care_submission_*'), key=lambda p: p.stat().st_mtime, reverse=True):
        if p.is_dir() and (p / 'Manuscript.docx').is_file():
            return p
        if p.suffix.lower() == '.zip' and zipfile.is_zipfile(p):
            return p
    return None


SUBMITTED = find_submitted()


def read_submitted(rel):
    """Bytes of a file of the submitted package by its path inside the package (for example 'Figures/Figure1_design_derivation.pdf')."""
    if SUBMITTED is None:
        return None
    if SUBMITTED.is_dir():
        p = SUBMITTED / rel
        return p.read_bytes() if p.is_file() else None
    with zipfile.ZipFile(SUBMITTED) as z:
        for n in z.namelist():
            if n == rel or n.endswith('/' + rel):
                return z.read(n)
    return None


STEP1 = RERUN_DERIVATION or not SAVED_RUN
STEP2 = RERUN_PIC or not PIC_DONE
if STEP1 and not (FREEZE / 'manifest.json').is_file():
    raise FileNotFoundError('There is no saved derivation run in this folder and AE_freeze_v1.4/ (with manifest.json) is missing.')
if STEP2 and not PIC_CONFIG.is_file():
    raise FileNotFoundError('The PIC outputs are missing and pic_external_validation/PIC_SUB_02_config.txt is not there: copy '
                            'PIC_SUB_02_config_template.txt to it and fill in the OSF link (and pic_dir if the PIC files are elsewhere).')

OUT.mkdir(exist_ok=True)
EXEC.mkdir(exist_ok=True)
for _p in OUT.iterdir():                                   # results of an earlier run of this notebook (executed copies are replaced)
    if _p.is_dir() and _p != EXEC:
        shutil.rmtree(_p)
    elif _p.is_file() and _p.suffix != '.ipynb':
        _p.unlink()
shutil.rmtree(WORK, ignore_errors=True)


def run_notebook(nb, python=None):
    """Runs a notebook in its own folder, as Run All does, and saves the executed copy in output_reproduction/executed_notebooks/.
    Returns the executed notebook (JSON)."""
    env = {k: v for k, v in os.environ.items() if k not in ('JPY_SESSION_NAME', 'JPY_PARENT_PID')}  # the child identifies its own file
    env['PYTHONIOENCODING'] = 'utf-8'
    cmd = [python or sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute', nb.name, '--output', nb.stem + '_executed',
           '--output-dir', str(EXEC), '--ExecutePreprocessor.timeout=-1', '--ExecutePreprocessor.kernel_name=python3']
    t0 = time.time()
    print(f"{time.strftime('%H:%M:%S')}  {nb.relative_to(HERE).as_posix()} ...", flush=True)
    r = subprocess.run(cmd, cwd=nb.parent, env=env, capture_output=True, text=True, encoding='utf-8', errors='replace')
    if r.returncode != 0:
        print(r.stderr[-6000:])
        raise RuntimeError(f'{nb.name} stopped with an error (above). Its own folder holds whatever it wrote before the error.')
    print(f'          finished in {(time.time() - t0) / 60:.1f} min', flush=True)
    return json.loads((EXEC / (nb.stem + '_executed.ipynb')).read_text(encoding='utf-8'))


def printed(nbjson):
    """Everything an executed notebook printed (stdout), as one text."""
    out = []
    for c in nbjson['cells']:
        for o in c.get('outputs', []):
            if o.get('output_type') == 'stream' and o.get('name') == 'stdout':
                out.append(''.join(o['text']) if isinstance(o['text'], list) else o['text'])
    return ''.join(out)


_status = [
    ('Derivation analysis: saved run (cache_figure_state/)', 'found' if SAVED_RUN else 'missing',
     'run it (hours)' if STEP1 else 'use it'),
    ('PIC analyses: outputs of PIC_SUB_02 and PIC_SUB_03', 'found' if PIC_DONE else 'missing',
     'run them (minutes)' if STEP2 else 'use them'),
    ('Figures 1-6 and S1-S6, Tables S1-S26', '', 'drawn by step 1' if STEP1 else 'redraw from the saved run'),
    ('Figure 7, PIC numbers, Tables S27-S33', '', 'PIC_SUB_04'),
    ('Visual abstract', '', 'build' if VISUAL_ABSTRACT else 'skip'),
    ('Submitted package', SUBMITTED.name if SUBMITTED else 'not found', 'compare' if SUBMITTED else 'no comparison'),
]
print(pd.DataFrame(_status, columns=['item', 'in this folder', 'action']).to_string(index=False))
print('\npython', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__, '| matplotlib',
      matplotlib.__version__, '| openpyxl', openpyxl.__version__, '| PIC steps with', PIC_PYTHON or 'this Python')

## Step 1. Derivation analysis (only when no saved run is in the folder)

In [ ]:
if STEP1:
    print('No saved run of the derivation analysis in this folder (or RERUN_DERIVATION is True): AE_subphenotypes_analysis.ipynb runs '
          'now. It takes hours; progress is written to output_subphenotypes/logs/progress.log.')
    run_notebook(ANALYSIS_NB)
else:
    print('Saved run of the derivation analysis found (cache_figure_state/): not run again.')
REDRAW = not STEP1                                               # a full run has just drawn the figures itself

## Step 2. PIC analyses (only when their outputs are missing)

In [ ]:
def own_marker_copies(nb):
    """Other notebooks in the PIC folder that carry the marker of nb: PIC_SUB_02 and PIC_SUB_03 identify their own file by it, so a
    second copy would leave their code fingerprint ambiguous."""
    m = re.search(r"NOTEBOOK_MARKER\s*=\s*'([^']+)'", nb.read_text(encoding='utf-8'))
    if not m:
        return []
    return [p.name for p in PIC.glob('*.ipynb') if p != nb and m.group(1) in p.read_text(encoding='utf-8', errors='ignore')]


if STEP2:
    print('PIC outputs missing (or RERUN_PIC is True): the PIC analyses run now.')
    for k in ('PIC_SUB_02_check', 'PIC_SUB_02_replication', 'PIC_SUB_03_posthoc'):
        extra = own_marker_copies(PIC_NBS[k])
        if extra:
            raise RuntimeError(f'Other copies of {k} in pic_external_validation/: {extra}. Keep only {PIC_NBS[k].name} there.')
        run_notebook(PIC_NBS[k], PIC_PYTHON)
else:
    print('Outputs of PIC_SUB_02 and PIC_SUB_03 found: not run again (the analysis of record is kept).')
for k in ('sub02_exact', 'sub03_exact'):
    _ri = json.loads((PIC_OUT[k] / 'run_info.json').read_text(encoding='utf-8'))
    print(f"{PIC_OUT[k].name}: code fingerprint {_ri.get('code_sha256', '')[:8]}..., synthetic fixture {_ri.get('synthetic_fixture')}, "
          f"python {_ri.get('python')}, pandas {_ri.get('pandas')}, numpy {_ri.get('numpy')}")

## Step 3. Figures 1-6 and S1-S6, Tables S1-S26 and the legends

In [ ]:
if REDRAW:
    run_notebook(REDRAW_NB)
FIGDIR = HERE / 'output_subphenotypes' / 'figures'
SUBDIR = HERE / 'output_subphenotypes' / 'submission'
FIG_STEMS = {}
for p in sorted(FIGDIR.glob('Figure*.pdf'), key=lambda p: p.stat().st_mtime):   # the newest file per number (renumbered files)
    m = re.match(r'Figure(S?\d+)_', p.name)
    if m:
        FIG_STEMS[m.group(1)] = p.stem
_need = [str(i) for i in range(1, 7)] + [f'S{i}' for i in range(1, 7)]
_lack = [k for k in _need if k not in FIG_STEMS or not (FIGDIR / (FIG_STEMS[k] + '.png')).is_file()]
_lack += [n for n in ('figure_legends.md',) if not (FIGDIR / n).is_file()]
_lack += [n for n in ('Additional_file_2_supplementary_tables.xlsx', 'Additional_file_3_supplementary_figures.pdf')
          if not (SUBDIR / n).is_file()]
if _lack:
    raise FileNotFoundError(f'Missing after step 3: {_lack}')
_wb = openpyxl.load_workbook(SUBDIR / 'Additional_file_2_supplementary_tables.xlsx', read_only=True)
print('Figures', ', '.join(FIG_STEMS[k] for k in _need[:6]))
print('Supplementary figures', ', '.join(FIG_STEMS[k] for k in _need[6:]))
print('Tables S1-S26:', _wb.sheetnames[1], 'to', _wb.sheetnames[-1], f'({len(_wb.sheetnames) - 1} tables and the contents sheet)')
_wb.close()

## Step 4. Figure 7, every PIC number of the text and Tables S27-S33

`PIC_SUB_04_manuscript_outputs.ipynb` runs in a temporary folder that holds copies of the PIC output tables, the frozen model, Tables
S1-S26 of step 3 and, when the submitted package is here, its `Manuscript.docx` (every PIC number of the text is then looked up in it).

In [ ]:
WORK.mkdir()
for _k, _src in PIC_OUT.items():
    if _src.is_dir():
        shutil.copytree(_src, WORK / _src.name)
shutil.copy2(MODEL, WORK / MODEL.name)
NB4 = WORK / PIC_NBS['PIC_SUB_04_manuscript_outputs'].name
shutil.copy2(PIC_NBS['PIC_SUB_04_manuscript_outputs'], NB4)
shutil.copy2(SUBDIR / 'Additional_file_2_supplementary_tables.xlsx', WORK / 'Additional_file_2_supplementary_tables.xlsx')
_docx = read_submitted('Manuscript.docx')
if _docx:
    (WORK / 'Manuscript.docx').write_bytes(_docx)
nb4 = run_notebook(NB4, PIC_PYTHON)
P4 = WORK / 'output_pic_sub04_manuscript'
INFO4 = json.loads((P4 / 'run_info_sub04.json').read_text(encoding='utf-8'))
TEXT4 = printed(nb4)
SUB04_LINES = [l for l in TEXT4.splitlines()
               if re.match(r'^(pic_numbers|manuscript_numbers|figure7_data|figure7_legend|tables_S27_S33)\s', l)
               or 'fragments found' in l or 'NOT FOUND' in l or 'manuscript check is skipped' in l]
print('\n'.join(SUB04_LINES))
print('All five results as in the manuscript build:', 'yes' if INFO4['matches_manuscript_build'] else 'NO')

## Step 5. Visual abstract

An editable 920 x 300 pixel slide (9.58 x 3.13 inches, so that PowerPoint's and LibreOffice's default export at 96 dpi gives 920 x
300 pixels) with every number read from the outputs above, and its PNG. Without LibreOffice, and on Windows without PowerPoint and
`pywin32`, only the slide is written: open it in PowerPoint and use *File > Export > PNG* (current slide).

In [ ]:
VA_NUMBERS = {}


def build_visual_abstract(out_pptx):
    """The visual abstract as a one-slide deck. Returns a dict of the numbers it shows."""
    from decimal import Decimal, ROUND_HALF_UP
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    from matplotlib.colors import LinearSegmentedColormap
    from matplotlib.patches import Rectangle as MRect
    from lxml import etree
    from pptx import Presentation
    from pptx.dml.color import RGBColor
    from pptx.enum.shapes import MSO_SHAPE, MSO_CONNECTOR
    from pptx.enum.text import PP_ALIGN, MSO_ANCHOR, MSO_AUTO_SIZE
    from pptx.util import Emu, Pt

    def r0(x):
        return str(Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

    # Numbers: frozen model, derivation tables, exact PIC tables and the PIC numbers of step 4
    TAB = HERE / 'output_subphenotypes' / 'tables'
    M = json.loads(MODEL.read_text(encoding='utf-8'))
    N = json.loads((P4 / 'pic_numbers.json').read_text(encoding='utf-8'))
    CLS = list(M['classes'])
    MU = np.asarray(M['means'], float)
    SHARE = pd.read_csv(TAB / 'class_summary.csv', encoding='utf-8-sig').set_index('class').loc[CLS].pct_modal.astype(float).values
    vb = pd.read_csv(TAB / 'validators_by_class.csv', encoding='utf-8-sig').set_index('validator')
    POOR = {c: float(str(vb.loc['Death or critical discharge', c]).split(' ')[0]) for c in ('S1', 'S6')}
    vc = pd.read_csv(TAB / 'validator_contrasts.csv', encoding='utf-8-sig')
    vc = vc[vc.contrast.eq('S1 vs S6')].set_index('validator')
    FOLD = {k: float(vc.loc[k, 'estimate']) for k in ('IL-6', 'IL-10', 'Ferritin')}
    cls_t = pd.read_csv(PIC_OUT['sub02_exact'] / '06_classes.csv', encoding='utf-8-sig', dtype=str).set_index('class').loc[CLS]
    R_PIC = cls_t.profile_r_vs_wuhan_same_class.astype(float).values
    sh = pd.read_csv(PIC_OUT['sub02'] / '13_sensitivity.csv', dtype=str, keep_default_na=False)
    R1 = sh[sh.analysis.str.startswith('Primary')].S1_profile_r_ci.iloc[0].split(' ')[0]
    MIN_R = float(N['_source']['run_info']['parameters']['MIN_R'])
    S1_PIC = N['classes']['S1']['pct']
    I10 = N['validators']['IL-10|S1 vs S6 (least abnormal)']['ratio_ci'][0]
    I6 = N['validators']['IL-6|S1 vs S6 (least abnormal)']['ratio_ci'][0]
    DEATH = {'S1': N['death']['S1']['pct_ci'][0], 'others': N['death']['others']['pct']}
    SENS, SPEC = N['bedside_pic']['sens'][0], N['bedside_pic']['spec'][0]
    CRIT = M['bedside_rule']['criteria']
    N_DER, N_PIC = int(M['model_children']), int(N['cohort']['n'])
    shown = {'derivation children': str(N_DER), 'PIC children': str(N_PIC),
             'subphenotype shares, derivation (%)': ', '.join(f'{v:.1f}' for v in SHARE),
             'S1 vs S6 ratios, derivation (IL-6, IL-10, ferritin)': ', '.join(r0(FOLD[k]) for k in ('IL-6', 'IL-10', 'Ferritin')),
             'death or critical discharge, S1 and S6 (%)': f"{r0(POOR['S1'])}, {r0(POOR['S6'])}",
             'S1 profile correlation in PIC; criterion': f'{R1}; {MIN_R:.2f}', 'S1 share in PIC (%)': f'{S1_PIC:.1f}',
             'S1 vs S6 ratios in PIC (IL-6, IL-10)': f'{I6:.1f}, {I10:.0f}',
             'in-hospital death in PIC, S1 and others (%)': f"{r0(DEATH['S1'])}, {r0(DEATH['others'])}",
             'bedside rule': '; '.join(f"{c['test'].upper()} >= {c['value']:g}" for c in CRIT) + f" (at least {M['bedside_rule']['positive_if_at_least']})",
             'rule sensitivity and specificity in PIC (%)': f'{r0(SENS)}, {r0(SPEC)}'}

    # Layout in design units: inches of a 9.2 x 3.0 inch slide at 100 dpi, scaled by 100/96 so that 96 dpi gives 920 x 300 pixels
    K = 100 / 96

    def E(v):
        return Emu(int(round(v * K * 914400)))

    def P(s):
        return Pt(s * K)

    RYB = ['#C62828', '#EF6C00', '#E3A600', '#2A9D8F', '#4FA3D9', '#1F3F8F']
    NAVY, BLUE, RED, TEAL, ORANGE, DARKRED, SLATE = '0F172A', '1F3F8F', 'C62828', '2A9D8F', 'EF6C00', '9B1C1C', '64748B'
    BG, WHITE, INK, MUTED, TRACK = 'F1F5F9', 'FFFFFF', '1E293B', '475569', 'E2E8F0'
    TINT = {'darkred': 'FDE8E8', 'teal': 'F0FDFA', 'blue': 'EFF6FF'}
    fonts = {f.name for f in font_manager.fontManager.ttflist}
    mpl_font = next((f for f in ['Arial', 'Liberation Sans'] if f in fonts), 'DejaVu Sans')
    rc = {'font.family': mpl_font, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.facecolor': 'white',
          'savefig.facecolor': 'white'}
    cmap = LinearSegmentedColormap.from_list('ryb_div', ['#1F3F8F', '#8FB3DA', '#F7F7F7', '#E9967A', '#C62828'])
    assets = WORK / 'visual_abstract_assets'
    assets.mkdir(exist_ok=True)

    with plt.rc_context(rc):                                      # mini plots at 300 dpi, at their size on the slide
        w_in, h_in = 2.66, 1.06
        fig = plt.figure(figsize=(w_in, h_in), dpi=300)
        lab_w, share_w = 0.30, 0.42
        ax = fig.add_axes([lab_w / w_in, 0.20, (w_in - lab_w - share_w) / w_in, 0.78])
        ax.imshow(np.clip(MU, -2, 2), cmap=cmap, vmin=-2, vmax=2, aspect='auto', interpolation='nearest')
        ax.add_patch(MRect((-0.5, -0.5), MU.shape[1], 1, fill=False, ec=RYB[0], lw=1.6, clip_on=False))
        ax.set_xticks([])
        ax.set_yticks(range(len(CLS)))
        ax.set_yticklabels(CLS, fontsize=6.4, fontweight='bold')
        for t, c in zip(ax.get_yticklabels(), RYB):
            t.set_color(c)
        ax.tick_params(axis='y', length=0, pad=1.5)
        for s_ in ax.spines.values():
            s_.set_visible(False)
        for k, v in enumerate(SHARE):
            ax.text(MU.shape[1] - 0.2, k, f' {v:.1f}%', va='center', ha='left', fontsize=6.4,
                    fontweight='bold' if k == 0 else 'normal', color=RYB[0] if k == 0 else '#334155')
        kx = fig.add_axes([lab_w / w_in, 0.035, 0.16, 0.06])
        kx.imshow(np.linspace(-2, 2, 64)[None, :], cmap=cmap, aspect='auto')
        kx.axis('off')
        fig.text(lab_w / w_in + 0.17, 0.065, f'low → high · {MU.shape[1]} tests and vital signs', fontsize=6.2, color='#475569',
                 va='center')
        fig.savefig(assets / 'heatmap.png', dpi=300)
        plt.close(fig)

        rw, rh = 1.70, 0.98
        fig = plt.figure(figsize=(rw, rh), dpi=300)
        ax = fig.add_axes([0.1, 0.2, 0.86, 0.6])
        ax.bar(range(len(CLS)), R_PIC, color=RYB, width=0.66)
        ax.axhline(MIN_R, color='#0F172A', lw=0.8, ls=(0, (3, 2)))
        ax.text(5.42, MIN_R + 0.03, f'criterion {MIN_R:.2f}', ha='right', va='bottom', fontsize=6, color='#0F172A')
        ax.text(0, R_PIC[0] + 0.03, R1, ha='center', va='bottom', fontsize=6.6, fontweight='bold', color=RYB[0])
        ax.set_xticks(range(len(CLS)))
        ax.set_xticklabels(CLS, fontsize=6.2, fontweight='bold')
        for t, c in zip(ax.get_xticklabels(), RYB):
            t.set_color(c)
        ax.set_ylim(0, 1.12)
        ax.set_yticks([0, 0.5, 1])
        ax.set_yticklabels(['0', '0.5', '1'], fontsize=5.6, color='#475569')
        ax.tick_params(length=2, width=0.6, pad=1.5)
        ax.tick_params(axis='x', length=0)
        for s_ in ('left', 'bottom'):
            ax.spines[s_].set_linewidth(0.6)
            ax.spines[s_].set_color('#94A3B8')
        fig.text(0.02, 0.97, 'Profile match with derivation (r)', fontsize=6.4, fontweight='bold', color='#' + TEAL, va='top')
        fig.savefig(assets / 'rbars.png', dpi=300)
        plt.close(fig)

    prs = Presentation()
    prs.slide_width, prs.slide_height = E(9.2), E(3.0)
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    A_NS = '{http://schemas.openxmlformats.org/drawingml/2006/main}'

    def rgb(h):
        return RGBColor.from_string(h)

    def box(x, y, w, h, fill, shape=MSO_SHAPE.ROUNDED_RECTANGLE, radius=None, line=None, line_w=0.75, name=None):
        s = slide.shapes.add_shape(shape, E(x), E(y), E(w), E(h))
        s.fill.solid()
        s.fill.fore_color.rgb = rgb(fill)
        if line:
            s.line.color.rgb = rgb(line)
            s.line.width = P(line_w)
        else:
            s.line.fill.background()
        s.shadow.inherit = False
        if radius is not None and shape == MSO_SHAPE.ROUNDED_RECTANGLE:
            s.adjustments[0] = radius
        if name:
            s.name = name
        return s

    def text(x, y, w, h, runs, size=9, color=INK, bold=False, align=PP_ALIGN.LEFT, anchor=MSO_ANCHOR.TOP, name=None):
        """runs: a string, or a list of paragraphs, each a list of (text, {size, color, bold}) runs."""
        tb = slide.shapes.add_textbox(E(x), E(y), E(w), E(h))
        tf = tb.text_frame
        tf.margin_left = tf.margin_right = tf.margin_top = tf.margin_bottom = 0
        tf.word_wrap = True
        tf.auto_size = MSO_AUTO_SIZE.NONE
        tf.vertical_anchor = anchor
        paras = [[(runs, {})]] if isinstance(runs, str) else runs
        for i, para in enumerate(paras):
            p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
            p.alignment = align
            for t, o in para:
                r = p.add_run()
                r.text = t
                r.font.name = 'Arial'
                r.font.size = P(o.get('size', size))
                r.font.bold = o.get('bold', bold)
                r.font.color.rgb = rgb(o.get('color', color))
        if name:
            tb.name = name
        return tb

    def picture(path, x, y, w, h, name):
        slide.shapes.add_picture(str(path), E(x), E(y), E(w), E(h)).name = name

    def line(x1, y1, x2, y2, color, width):
        c = slide.shapes.add_connector(MSO_CONNECTOR.STRAIGHT, E(x1), E(y1), E(x2), E(y2))
        c.line.color.rgb = rgb(color)
        c.line.width = P(width)
        return c

    def ring(cx, cy, d, pct, color, track, thick=0.2):
        """A proportional ring: a light donut and, over it, a block arc of pct% clockwise from 12 o'clock."""
        for shape, fill in ((MSO_SHAPE.DONUT, track), (MSO_SHAPE.BLOCK_ARC, color)):
            s = box(cx - d / 2, cy - d / 2, d, d, fill, shape=shape)
            av = s._element.spPr.find(A_NS + 'prstGeom').find(A_NS + 'avLst')
            for g in list(av):
                av.remove(g)
            vals = ([('adj', f'val {round(thick * 100000)}')] if shape == MSO_SHAPE.DONUT else
                    [('adj1', f'val {round(270 * 60000)}'), ('adj2', f'val {round(((270 + 360 * pct / 100) % 360) * 60000)}'),
                     ('adj3', f'val {round(thick * 100000)}')])
            for nm, v in vals:
                g = etree.SubElement(av, A_NS + 'gd')
                g.set('name', nm)
                g.set('fmla', v)

    def hbar(x, y, w, label, pct, color):
        text(x, y - 0.012, 0.36, 0.14, label, size=7.5, bold=True, color=color)
        bx, bw = x + 0.40, w - 0.82
        box(bx, y, bw, 0.12, TRACK, radius=0.5)
        box(bx, y, max(bw * pct / 100, 0.02), 0.12, color, radius=0.5)
        text(bx + bw * pct / 100 + 0.05, y - 0.02, 0.4, 0.16, f'{r0(pct)}%', size=8.5, bold=True, color=color)

    def icon_hospital(x, y, s=0.44):
        box(x, y, s, s, 'E0F2EF', shape=MSO_SHAPE.OVAL, line='A8D8CF', name='Icon hospital')
        box(x + 0.12, y + 0.15, 0.20, 0.20, WHITE, radius=0.08, line=TEAL, line_w=0.9)
        box(x + 0.172, y + 0.065, 0.096, 0.096, WHITE, radius=0.18, line=TEAL, line_w=0.9)
        box(x + 0.189, y + 0.104, 0.062, 0.018, 'D32F2F', shape=MSO_SHAPE.RECTANGLE)
        box(x + 0.211, y + 0.082, 0.018, 0.062, 'D32F2F', shape=MSO_SHAPE.RECTANGLE)
        for dx in (0.15, 0.255):
            for dy in (0.195, 0.245):
                box(x + dx, y + dy, 0.035, 0.032, TEAL, shape=MSO_SHAPE.RECTANGLE)
        box(x + 0.2, y + 0.29, 0.04, 0.06, TEAL, shape=MSO_SHAPE.RECTANGLE)

    def icon_antibody(x, y, s=0.5):
        box(x, y, s, s, 'E3EEFC', shape=MSO_SHAPE.OVAL, line='B9D2F3', name='Icon antibody')
        cx, cy = x + s / 2, y + s / 2
        for sx in (-1, 1):                                        # light chains, beside the outer half of each arm
            line(cx + sx * 0.077, cy + 0.023, cx + sx * 0.152, cy - 0.052, '4FA3D9', 1.8)
        line(cx, cy + 0.03, cx, cy + 0.18, BLUE, 2.6)                         # heavy chains: stem and arms
        for sx in (-1, 1):
            line(cx, cy + 0.03, cx + sx * 0.12, cy - 0.09, BLUE, 2.6)
        for sx in (-1, 1):                                        # antigen-binding tips
            box(cx + sx * 0.12 - 0.033, cy - 0.09 - 0.033, 0.066, 0.066, 'D32F2F', shape=MSO_SHAPE.OVAL, line=WHITE, line_w=0.75)

    box(0, 0, 9.2, 3.0, BG, shape=MSO_SHAPE.RECTANGLE, name='Background')
    box(0.10, 0.08, 9.0, 0.42, NAVY, radius=0.18, name='Title band')
    text(0.24, 0.08, 8.8, 0.42, 'Routine Admission Tests Identify a Hyperinflammatory Subphenotype of Childhood Acute Encephalopathy',
         size=11.5, bold=True, color=WHITE, anchor=MSO_ANCHOR.MIDDLE, name='Title')
    CARD_Y, CARD_H, CARD_W = 0.58, 2.34, 2.88
    CX = [0.10, 3.16, 6.22]
    HEAD = [(BLUE, f'1  DERIVATION  (n = {N_DER})'), (TEAL, f'2  EXTERNAL VALIDATION  (n = {N_PIC})'), (RED, '3  BEDSIDE RULE')]
    for i, (x0, (col, lab)) in enumerate(zip(CX, HEAD)):
        box(x0, CARD_Y, CARD_W, CARD_H, WHITE, radius=0.04, name=f'Card {i + 1}')
        box(x0 + 0.06, CARD_Y + 0.06, CARD_W - 0.12, 0.25, col, radius=0.2, name=f'Card {i + 1} header')
        text(x0 + 0.14, CARD_Y + 0.06, CARD_W - 0.2, 0.25, lab, size=8.5, bold=True, color=WHITE, anchor=MSO_ANCHOR.MIDDLE)
        if i < 2:
            box(x0 + CARD_W + 0.03, CARD_Y + CARD_H / 2 - 0.1, 0.12, 0.2, TEAL if i == 0 else RED, shape=MSO_SHAPE.RIGHT_ARROW)

    x0 = CX[0]                                                    # 1. derivation
    picture(assets / 'heatmap.png', x0 + 0.11, CARD_Y + 0.36, 2.66, 1.06, 'Subphenotype heatmap')
    box(x0 + 0.11, CARD_Y + 1.46, 2.66, 0.27, TINT['blue'], radius=0.25)
    text(x0 + 0.11, CARD_Y + 1.46, 2.66, 0.27, [[('S1 vs S6:  ', {'size': 8, 'color': MUTED}),
                                                   (f"IL-6 ×{r0(FOLD['IL-6'])}", {'size': 9, 'bold': True, 'color': RED}), ('   ', {}),
                                                   (f"IL-10 ×{r0(FOLD['IL-10'])}", {'size': 9, 'bold': True, 'color': ORANGE}),
                                                   ('   ', {}),
                                                   (f"Ferritin ×{r0(FOLD['Ferritin'])}", {'size': 9, 'bold': True, 'color': DARKRED})]],
         align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE)
    text(x0 + 0.11, CARD_Y + 1.80, 2.7, 0.15, 'Death or critical discharge (S1 vs S6)', size=7.5, bold=True)
    hbar(x0 + 0.11, CARD_Y + 1.99, 2.7, 'S1', POOR['S1'], RED)
    hbar(x0 + 0.11, CARD_Y + 2.15, 2.7, 'S6', POOR['S6'], BLUE)

    x0 = CX[1]                                                    # 2. external validation
    picture(assets / 'rbars.png', x0 + 0.08, CARD_Y + 0.38, 1.70, 0.98, 'Profile correlation by subphenotype')
    icon_hospital(x0 + 1.98, CARD_Y + 0.42)
    text(x0 + 1.84, CARD_Y + 0.90, 0.98, 0.3, [[(f'{S1_PIC:.1f}%', {'size': 15, 'bold': True, 'color': RED})]], align=PP_ALIGN.CENTER)
    text(x0 + 1.80, CARD_Y + 1.16, 1.06, 0.18, 'hyperinflammatory', size=7.5, bold=True, color=RED, align=PP_ALIGN.CENTER)
    box(x0 + 0.11, CARD_Y + 1.46, 2.66, 0.27, TINT['teal'], radius=0.25)
    text(x0 + 0.11, CARD_Y + 1.46, 2.66, 0.27, [[('S1 vs S6:  ', {'size': 8, 'color': MUTED}),
                                                   (f'IL-6 ×{I6:.1f}', {'size': 9, 'bold': True, 'color': RED}), ('   ', {}),
                                                   (f'IL-10 ×{I10:.0f}', {'size': 9, 'bold': True, 'color': ORANGE})]],
         align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE)
    text(x0 + 0.11, CARD_Y + 1.80, 2.7, 0.15, 'In-hospital death (S1 vs other subphenotypes)', size=7.5, bold=True)
    hbar(x0 + 0.11, CARD_Y + 1.99, 2.7, 'S1', DEATH['S1'], RED)
    hbar(x0 + 0.11, CARD_Y + 2.15, 2.7, 'Others', DEATH['others'], SLATE)

    x0 = CX[2]                                                    # 3. bedside rule
    for j, c in enumerate(CRIT):
        cx = x0 + 0.11 + j * 0.90
        unit = '' if c['unit'] == 'ratio' else f" {c['unit']}"
        box(cx, CARD_Y + 0.38, 0.84, 0.44, TINT['darkred'], radius=0.14)
        text(cx, CARD_Y + 0.40, 0.84, 0.22, c['test'].upper(), size=10, bold=True, color=RED, align=PP_ALIGN.CENTER)
        text(cx, CARD_Y + 0.61, 0.84, 0.18, f"≥ {c['value']:g}{unit}", size=7.5, align=PP_ALIGN.CENTER)
    k_rule = M['bedside_rule']['positive_if_at_least']
    text(x0 + 0.11, CARD_Y + 0.88, 2.66, 0.17, f'At least {k_rule} of {len(CRIT)} positive at admission', size=8, bold=True,
         align=PP_ALIGN.CENTER)
    for j, (v, lab, col, track) in enumerate([(SENS, 'Sensitivity', RED, 'F6D5D5'), (SPEC, 'Specificity', TEAL, 'CDEBE7')]):
        cx, cy, d = x0 + 0.52 + j * 0.92, CARD_Y + 1.42, 0.62
        ring(cx, cy, d, v, col, track)
        text(cx - 0.3, cy - 0.09, 0.6, 0.18, f'{r0(v)}%', size=9.5, bold=True, color=col, align=PP_ALIGN.CENTER,
             anchor=MSO_ANCHOR.MIDDLE)
        text(cx - 0.45, cy + 0.33, 0.9, 0.15, lab, size=7.5, bold=True, align=PP_ALIGN.CENTER)
    text(x0 + 0.11, CARD_Y + 1.98, 1.85, 0.14, 'in the validation cohort (PIC)', size=7, color=MUTED, align=PP_ALIGN.CENTER)
    icon_antibody(x0 + 2.14, CARD_Y + 1.18)
    text(x0 + 1.96, CARD_Y + 1.72, 0.86, 0.4, [[('Trial selection', {'size': 7.5, 'bold': True, 'color': BLUE})],
                                                [('IL-6 / IL-1 blockade', {'size': 7})]], align=PP_ALIGN.CENTER)

    import datetime
    cp = prs.core_properties                                      # not the metadata of python-pptx's template
    cp.title, cp.author, cp.last_modified_by, cp.revision = 'Visual Abstract', '', '', 1
    cp.created = cp.modified = datetime.datetime.now().replace(microsecond=0)
    prs.save(out_pptx)
    tmp = out_pptx.with_suffix('.tmp')                            # flat shapes: no theme shadow (some renderers draw one)
    with zipfile.ZipFile(out_pptx) as zin, zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            data = zin.read(item.filename)
            if item.filename.startswith('ppt/slides/slide') and item.filename.endswith('.xml'):
                data = re.sub(rb'<a:effectRef idx="[1-9]">', b'<a:effectRef idx="0">', data)
            zout.writestr(item, data)
    shutil.move(str(tmp), str(out_pptx))
    return shown


def render_png(pptx, png):
    """PNG of the slide at its default export size (920 x 300): LibreOffice if installed, else PowerPoint on Windows (pywin32)."""
    cands = [shutil.which('soffice'), shutil.which('soffice.exe'), r'C:\Program Files\LibreOffice\program\soffice.exe',
             r'C:\Program Files (x86)\LibreOffice\program\soffice.exe', '/Applications/LibreOffice.app/Contents/MacOS/soffice']
    soffice = next((c for c in cands if c and Path(c).is_file()), None)
    if soffice:
        tmpdir = WORK / 'visual_abstract_render'
        tmpdir.mkdir(exist_ok=True)
        subprocess.run([soffice, '--headless', '--convert-to', 'png', '--outdir', str(tmpdir), str(pptx)], check=True,
                       capture_output=True, timeout=300)
        shutil.move(str(tmpdir / (pptx.stem + '.png')), str(png))
        return 'LibreOffice'
    try:
        import win32com.client
        app = win32com.client.Dispatch('PowerPoint.Application')
        pres = app.Presentations.Open(str(pptx.resolve()), True, False, False)      # read-only, no window
        pres.Slides(1).Export(str(png.resolve()), 'PNG', 920, 300)
        pres.Close()
        if app.Presentations.Count == 0:
            app.Quit()
        return 'PowerPoint'
    except Exception:
        return None


VA_PPTX, VA_PNG, VA_HOW = OUT / 'Visual_Abstract_920x300.pptx', OUT / 'Visual Abstract.png', None
if VISUAL_ABSTRACT:
    try:
        import pptx  # noqa: F401
        HAVE_PPTX = True
    except ImportError:
        HAVE_PPTX = False
        print('python-pptx is not installed (pip install python-pptx): the visual abstract is skipped.')
    if HAVE_PPTX:
        VA_NUMBERS = build_visual_abstract(VA_PPTX)
        VA_HOW = render_png(VA_PPTX, VA_PNG)
        print(pd.Series(VA_NUMBERS).to_string())
        if VA_HOW:
            from PIL import Image
            _sz = Image.open(VA_PNG).size
            print(f'\n{VA_PNG.name}: {_sz[0]} x {_sz[1]} pixels, {VA_PNG.stat().st_size / 1024:.0f} KB (rendered by {VA_HOW}; the journal '
                  'asks for 920 x 300 pixels and at most 150 KB)')
        else:
            print(f'\n{VA_PPTX.name} written; no LibreOffice or PowerPoint found for the PNG: open it in PowerPoint and use File > '
                  'Export > PNG (current slide); the default export is 920 x 300 pixels.')

## Step 6. Collection in `output_reproduction/`

Additional file 2 is laid out as three-line tables (a heavy rule above the header, a light rule below it and a heavy rule below the
last row; no other lines). Only the layout changes: every value stays as the notebooks wrote it, except that five numbers written in
E notation by the notebook's formatting (Tables S8 and S9, for example 1.54e+03) are written out (1540).

In [ ]:
def format_three_line(path, font='Arial', size=10):
    """Formats every sheet of an Excel workbook as three-line tables, in place: a heavy rule above the header row, a light rule
    below it and a heavy rule below the last row, with no other lines and no gridlines. The sheet title (A1), subheadings and
    header rows are bold and notes italic. Decimal numbers keep their full precision and are displayed with up to three decimals
    (scientific notation below 0.001, thousands separators from 1,000). In text cells, a number written in E notation with a
    positive exponent (for example '1.54e+03', three significant digits from the notebook's formatting) is written out ('1540').
    A table is a run of non-blank rows whose first row is its header; a single-cell row directly above a header is its subheading;
    a single-cell row on its own is a note."""
    import re
    import openpyxl
    from openpyxl.styles import Alignment, Border, Font, Side
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.properties import PageSetupProperties

    heavy, light = Side(style='medium', color='000000'), Side(style='thin', color='000000')
    numeric_text = re.compile(r'^\s*[-<>≤≥]?\s*\d')
    e_notation = re.compile(r'(?<![\w.])\d\.\d+e\+0*[3-9]\b')        # 1.54e+03 and the like: whole numbers of 1,000 or more
    wb = openpyxl.load_workbook(path)
    for ws in wb.worksheets:
        ws.sheet_view.showGridLines = False
        ws.page_setup.orientation = 'landscape'
        ws.sheet_properties.pageSetUpPr = PageSetupProperties(fitToPage=True)
        ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
        ncol = ws.max_column

        def filled(r):
            return [c for c in range(1, ncol + 1) if ws.cell(r, c).value not in (None, '')]

        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ncol):
            for cell in row:
                cell.font, cell.border, cell.alignment = Font(name=font, size=size), Border(), Alignment()
                if isinstance(cell.value, str) and e_notation.search(cell.value):
                    cell.value = e_notation.sub(lambda m: f'{float(m.group(0)):f}'.rstrip('0').rstrip('.'), cell.value)
        ws.cell(1, 1).font = Font(name=font, size=size + 1, bold=True)
        blocks, cur = [], []
        for r in range(2, ws.max_row + 1):
            if filled(r):
                cur.append(r)
            elif cur:
                blocks.append(cur)
                cur = []
        if cur:
            blocks.append(cur)
        width_chars = {}
        for b in blocks:
            first = filled(b[0])
            if len(b) == 1 and len(first) == 1:                   # a note
                ws.cell(b[0], first[0]).font = Font(name=font, size=size, italic=True)
                continue
            sub = len(first) == 1 and len(filled(b[1])) >= 2
            if sub:                                               # subheading of the table below it
                ws.cell(b[0], first[0]).font = Font(name=font, size=size, bold=True)
            body = b[1:] if sub else b
            head, data = body[0], body[1:]
            ncols = max(max(filled(r)) for r in body)
            for c in range(1, ncols + 1):
                vals = [ws.cell(r, c).value for r in data]
                vals = [v for v in vals if v not in (None, '')]
                numeric = sum((isinstance(v, (int, float)) and not isinstance(v, bool)) or
                              (isinstance(v, str) and bool(numeric_text.match(v))) for v in vals)
                align = 'left' if c == 1 or (vals and numeric < 0.6 * len(vals)) else 'center'
                h = ws.cell(head, c)
                h.font = Font(name=font, size=size, bold=True)
                h.border = Border(top=heavy, bottom=light if data else heavy)
                h.alignment = Alignment(horizontal=align, vertical='bottom', wrap_text=True)
                for r in data:
                    x = ws.cell(r, c)
                    x.border = Border(bottom=heavy) if r == data[-1] else Border()
                    v = x.value
                    if isinstance(v, float):
                        x.number_format = ('0' if v == 0 else '0.00E+00' if abs(v) < 0.001 else
                                           '#,##0.0' if abs(v) >= 1000 else '0.0##')
                        shown = len(f'{v:.3g}') + 2
                    else:
                        shown = len(str(v)) if v is not None else 0
                    x.alignment = Alignment(horizontal=align, vertical='top', wrap_text=shown > 48)
                    width_chars[c] = max(width_chars.get(c, 0), min(shown, 48))
                longest_word = max((len(w) for w in str(h.value or '').split()), default=0)
                width_chars[c] = max(width_chars.get(c, 0), min(longest_word, 48), 6)
        for c, w in width_chars.items():
            ws.column_dimensions[get_column_letter(c)].width = w + 2
    wb.save(path)



(OUT / 'Figures').mkdir()
(OUT / 'Supplementary_figures').mkdir()
for k in [str(i) for i in range(1, 7)]:
    for ext in ('.pdf', '.png'):
        shutil.copy2(FIGDIR / (FIG_STEMS[k] + ext), OUT / 'Figures' / (FIG_STEMS[k] + ext))
for ext in ('.pdf', '.png'):
    shutil.copy2(P4 / ('Figure7_external_validation' + ext), OUT / 'Figures' / ('Figure7_external_validation' + ext))
for k in [f'S{i}' for i in range(1, 7)]:
    for ext in ('.pdf', '.png'):
        shutil.copy2(FIGDIR / (FIG_STEMS[k] + ext), OUT / 'Supplementary_figures' / (FIG_STEMS[k] + ext))
shutil.copy2(P4 / 'Additional_file_2_supplementary_tables.xlsx', OUT / 'Additional_file_2_supplementary_tables.xlsx')
format_three_line(OUT / 'Additional_file_2_supplementary_tables.xlsx')
shutil.copy2(SUBDIR / 'Additional_file_3_supplementary_figures.pdf', OUT / 'Additional_file_3_supplementary_figures.pdf')
for n in ('pic_numbers.json', 'manuscript_numbers.csv', 'run_info_sub04.json'):
    shutil.copy2(P4 / n, OUT / n)


def legend_paragraphs(text):
    """{'1': 'Figure 1. ...', 'S1': ...} from a Markdown file of legends (one paragraph per legend)."""
    out = {}
    for para in re.split(r'\n\s*\n', text):
        para = ' '.join(para.split())
        m = re.match(r'^\**Figure (S?\d+)\.', para)
        if m:
            out[m.group(1)] = para
    return out


LEG = legend_paragraphs((FIGDIR / 'figure_legends.md').read_text(encoding='utf-8-sig'))
LEG['7'] = ' '.join((P4 / 'figure7_legend.md').read_text(encoding='utf-8-sig').split())
_order = [str(i) for i in range(1, 8)] + [f'S{i}' for i in range(1, 7)]
(OUT / 'figure_legends.md').write_text('# Figure legends\n\n' + '\n\n'.join(LEG[k] for k in _order if k in LEG) + '\n',
                                       encoding='utf-8')
for p in sorted(OUT.rglob('*')):
    if p.is_file() and EXEC not in p.parents and p.suffix != '.ipynb':
        print(f'{p.relative_to(OUT).as_posix():60s} {p.stat().st_size / 1024:8.0f} KB')

## Step 7. Comparison with the submitted package

Tables: every cell of Tables S1-S33 and the contents sheet (numbers equal within a relative difference of 10^-9, so differences in
the last digits of floating-point results do not count). Legends: the text of each legend against the manuscript's. Figures: the PDF
files, apart from their creation date (other software versions or fonts change the file without changing the content; the
figure is then to be compared by eye: with PyMuPDF installed, `pip install pymupdf`, each such figure is also rendered next to the
submitted one in `comparison/figures/`). Visual abstract: pixel by pixel.

In [ ]:
RESAMPLING_COLUMNS = {'converged', 'boot_used', 'boot_not_converged', 'delta_c_ci', 'simulations', 'p_parametric_bootstrap'}
RESAMPLING_WORDS = ('bootstrap', 'simulation', 'converge', 'resampl', 'refit')
NUM = re.compile(r'-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?')


def same_value(a, b):
    a = None if a == '' else a
    b = None if b == '' else b
    if a is None or b is None:
        return a is None and b is None
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)
    sa, sb = str(a), str(b)
    if sa == sb:
        return True
    if NUM.sub('#', sa) != NUM.sub('#', sb):
        return False
    xa, xb = [float(x) for x in NUM.findall(sa)], [float(x) for x in NUM.findall(sb)]
    return len(xa) == len(xb) and all(math.isclose(x, y, rel_tol=1e-9, abs_tol=1e-12) for x, y in zip(xa, xb))


def docx_paragraphs(data):
    """Paragraph texts of a .docx file (without python-docx)."""
    W = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        root = ET.fromstring(z.read('word/document.xml'))
    return [''.join(t.text or '' for t in p.iter(W + 't')) for p in root.iter(W + 'p')]


def norm(s):
    s = s.translate(str.maketrans({'\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"'})).replace('**', '')
    return ' '.join(s.split())


ROWS = []                                                        # (part, item, result, detail)
TABLE_DIFFS = []
_need_sub = ['Additional_file_2_supplementary_tables.xlsx', 'Manuscript.docx']
if SUBMITTED is not None and any(read_submitted(n) is None for n in _need_sub):
    print(f'{SUBMITTED.name} lacks {[n for n in _need_sub if read_submitted(n) is None]}: no comparison.')
    SUBMITTED = None
if SUBMITTED is None:
    print('No submitted package to compare with (Critical_Care_submission_*.zip or its folder, next to this notebook).')
else:
    (OUT / 'comparison').mkdir()
    # Tables S1-S33
    wb_r = openpyxl.load_workbook(OUT / 'Additional_file_2_supplementary_tables.xlsx', data_only=True)
    wb_s = openpyxl.load_workbook(io.BytesIO(read_submitted('Additional_file_2_supplementary_tables.xlsx')), data_only=True)
    for name in sorted(set(wb_r.sheetnames) | set(wb_s.sheetnames), key=lambda n: (n != 'Contents', int(re.sub(r'\D', '', n) or 0))):
        if name not in wb_r.sheetnames or name not in wb_s.sheetnames:
            ROWS.append(('Tables', name, 'DIFFERENT', 'sheet missing in the ' + ('reproduction' if name not in wb_r.sheetnames else 'submission')))
            continue
        a = [list(r) for r in wb_r[name].iter_rows(values_only=True)]
        b = [list(r) for r in wb_s[name].iter_rows(values_only=True)]
        nr, nc = max(len(a), len(b)), max([len(r) for r in a + b] or [0])
        hdr_a, hdr_b = (a[2] if len(a) > 2 else []), (b[2] if len(b) > 2 else [])   # header row of single-table sheets
        diffs = []
        for i in range(nr):
            for j in range(nc):
                va = a[i][j] if i < len(a) and j < len(a[i]) else None
                vb = b[i][j] if i < len(b) and j < len(b[i]) else None
                if not same_value(va, vb):
                    col = str((hdr_a[j] if j < len(hdr_a) else None) or (hdr_b[j] if j < len(hdr_b) else None) or '')
                    first = str((a[i][0] if i < len(a) and a[i] else '') or (b[i][0] if i < len(b) and b[i] else ''))
                    kind = ('resampling-dependent' if col in RESAMPLING_COLUMNS or any(w in (col + ' ' + first).lower()
                                                                                        for w in RESAMPLING_WORDS) else 'other')
                    diffs.append((name, f'{openpyxl.utils.get_column_letter(j + 1)}{i + 1}', col or first, va, vb, kind))
        TABLE_DIFFS += diffs
        if not diffs:
            ROWS.append(('Tables', name, 'same', ''))
        else:
            n_other = sum(d[5] == 'other' for d in diffs)
            ROWS.append(('Tables', name, 'DIFFERENT' if n_other else 'resampling-dependent',
                         f'{len(diffs)} cells' + (f', {n_other} not resampling-dependent' if n_other else '')))
    # Legends
    paras = docx_paragraphs(read_submitted('Manuscript.docx'))
    i0 = next(i for i, t in enumerate(paras) if t.strip() == 'Figure legends')
    i1 = next(i for i, t in enumerate(paras) if i > i0 and t.strip() == 'Additional files')
    LEG_S = {m.group(1): norm(t) for t in paras[i0 + 1:i1] if (m := re.match(r'\s*Figure (\d+)\.', t))}
    for k in [str(i) for i in range(1, 8)]:
        r_, s_ = norm(LEG.get(k, '')), LEG_S.get(k, '')
        if r_ == s_:
            ROWS.append(('Legends', f'Figure {k}', 'same', ''))
            continue
        wa, wb = r_.split(' '), s_.split(' ')
        segs, kinds = [], []
        for op, a1, a2, b1, b2 in difflib.SequenceMatcher(None, wa, wb, autojunk=False).get_opcodes():
            if op == 'equal':
                continue
            ctx = ' '.join(wa[max(0, a1 - 6):a1])
            segs.append(f"... {ctx} [reproduced: {' '.join(wa[a1:a2]) or '-'} | submitted: {' '.join(wb[b1:b2]) or '-'}]")
            kinds.append(any(w in (ctx + ' ' + ' '.join(wa[a1:a2] + wb[b1:b2])).lower() for w in RESAMPLING_WORDS))
        ROWS.append(('Legends', f'Figure {k}', 'resampling-dependent' if all(kinds) else 'DIFFERENT', ' '.join(segs)))
    # Figures (PDF), apart from the creation date
    try:                                                         # PyMuPDF (optional): pixel comparison and side-by-side images
        import pymupdf as fitz
    except ImportError:
        try:
            import fitz
        except ImportError:
            fitz = None
    if fitz is None:
        print('PyMuPDF is not installed (pip install pymupdf): figures that differ as files are listed for comparison by eye.')
    for p in sorted((OUT / 'Figures').glob('*.pdf')):
        sub = read_submitted('Figures/' + p.name)
        if sub is None:
            ROWS.append(('Figures', p.stem, 'not in the submission', ''))
            continue
        strip = lambda b: re.sub(rb'/CreationDate \(D:[^)]*\)', b'', b)
        if strip(p.read_bytes()) == strip(sub):
            ROWS.append(('Figures', p.stem, 'same', 'identical file apart from its creation date'))
            continue
        detail = 'the files differ (software versions or fonts): compare by eye'
        if fitz is not None:
            from PIL import Image

            def img(b):
                pix = fitz.open(stream=b, filetype='pdf')[0].get_pixmap(dpi=72)
                return np.frombuffer(pix.samples, np.uint8).reshape(pix.h, pix.w, pix.n)[..., :3].copy()
            ia, ib = img(p.read_bytes()), img(sub)
            h, w = min(ia.shape[0], ib.shape[0]), min(ia.shape[1], ib.shape[1])
            if max(abs(ia.shape[0] - ib.shape[0]), abs(ia.shape[1] - ib.shape[1])) <= 2:
                share = 100 * float((np.abs(ia[:h, :w].astype(int) - ib[:h, :w].astype(int)).max(axis=2) > 48).mean())
                detail += f'; {share:.2f}% of pixels differ at 72 dpi (a shift of a fraction of a point changes edges throughout)'
            else:
                detail += f'; page sizes differ ({ia.shape[1]} x {ia.shape[0]} vs {ib.shape[1]} x {ib.shape[0]} pixels at 72 dpi)'
            (OUT / 'comparison' / 'figures').mkdir(parents=True, exist_ok=True)
            pair = np.full((max(ia.shape[0], ib.shape[0]), ia.shape[1] + ib.shape[1] + 20, 3), 255, np.uint8)
            pair[:ia.shape[0], :ia.shape[1]] = ia
            pair[:ib.shape[0], ia.shape[1] + 20:] = ib
            Image.fromarray(pair).save(OUT / 'comparison' / 'figures' / f'{p.stem}_reproduced_left_submitted_right.png')
            detail += '; side by side in comparison/figures/'
        ROWS.append(('Figures', p.stem, 'check by eye', detail))
    # Visual abstract
    sub_va = read_submitted('Visual Abstract.png')
    if VA_HOW and sub_va:
        from PIL import Image
        ia = np.asarray(Image.open(VA_PNG).convert('RGB'), int)
        ib = np.asarray(Image.open(io.BytesIO(sub_va)).convert('RGB'), int)
        if ia.shape != ib.shape:
            ROWS.append(('Visual abstract', 'Visual Abstract.png', 'check by eye', f'sizes differ: {ia.shape[1]} x {ia.shape[0]} vs {ib.shape[1]} x {ib.shape[0]}'))
        else:
            share = 100 * float((np.abs(ia - ib).max(axis=2) > 48).mean())
            ROWS.append(('Visual abstract', 'Visual Abstract.png', 'same' if share < 0.5 else 'check by eye',
                         f'{share:.2f}% of pixels differ (fonts and anti-aliasing of the renderer)'))
    # PIC numbers (PIC_SUB_04 against the manuscript build) and the PIC numbers in Manuscript.docx
    for l in SUB04_LINES:
        m = re.match(r'^(\S+)\s+(MATCH|DIFFERENT)', l)
        if m:
            ROWS.append(('PIC_SUB_04', m.group(1), 'same' if m.group(2) == 'MATCH' else 'DIFFERENT', 'against the manuscript build'))
        elif 'fragments found' in l:
            ROWS.append(('PIC_SUB_04', 'PIC numbers in Manuscript.docx', 'same' if re.search(r'(\d+) of \1 ', l + ' ') else 'DIFFERENT', l.strip()))

    CMP = pd.DataFrame(ROWS, columns=['part', 'item', 'result', 'detail'])
    CMP.to_csv(OUT / 'comparison' / 'comparison_with_submission.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(TABLE_DIFFS, columns=['sheet', 'cell', 'column or row', 'reproduced', 'submitted', 'kind']).to_csv(
        OUT / 'comparison' / 'table_cells_that_differ.csv', index=False, encoding='utf-8-sig')
    with pd.option_context('display.max_colwidth', 160, 'display.width', 250):
        print(CMP.groupby(['part', 'result']).size().to_string(), '\n')
        print(CMP[CMP.result != 'same'].to_string(index=False) if (CMP.result != 'same').any() else 'Everything is the same as in the submission.')
        if TABLE_DIFFS:
            print(f'\nTable cells that differ ({len(TABLE_DIFFS)}; the first 60 below, all in comparison/table_cells_that_differ.csv):')
            print(pd.DataFrame(TABLE_DIFFS[:60], columns=['sheet', 'cell', 'column or row', 'reproduced', 'submitted', 'kind']).to_string(index=False))

## Summary

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)                          # the temporary copies of the PIC tables
files = {p.relative_to(OUT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
         for p in sorted(OUT.rglob('*')) if p.is_file() and EXEC not in p.parents and p.suffix != '.ipynb'}
info = {'notebook': 'AE_reproduce_figures_tables', 'finished': time.strftime('%Y-%m-%d %H:%M:%S'),
        'minutes': round((time.time() - T_START) / 60, 1),
        'steps_run': {'derivation_analysis': STEP1, 'pic_analyses': STEP2, 'redraw': REDRAW, 'pic_sub04': True,
                      'visual_abstract': VA_HOW or (bool(VA_NUMBERS) and 'slide only') or False},
        'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
        'pic_python': PIC_PYTHON or 'same', 'pic_sub04': INFO4, 'submitted_package': SUBMITTED.name if SUBMITTED else None,
        'comparison': (CMP.groupby('result').size().to_dict() if SUBMITTED is not None else None), 'files_sha256': files}
(OUT / 'run_info_reproduction.json').write_text(json.dumps(info, indent=1, default=str), encoding='utf-8')
print(f"Done in {info['minutes']} min. Figures 1-7: output_reproduction/Figures; Figures S1-S6: Supplementary_figures and "
      "Additional_file_3_supplementary_figures.pdf; Tables S1-S33: Additional_file_2_supplementary_tables.xlsx; legends: "
      "figure_legends.md; PIC numbers: pic_numbers.json and manuscript_numbers.csv; visual abstract: Visual Abstract.png and "
      "Visual_Abstract_920x300.pptx.")
if SUBMITTED is not None:
    print('Comparison with', SUBMITTED.name + ':', ', '.join(f'{k} {v}' for k, v in info['comparison'].items()),
          '(details in output_reproduction/comparison/).')
print('output_reproduction/ holds aggregate results only. The dataset, the PIC files, cache_* and *_LOCAL_ONLY folders stay on this computer.')